In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

CATEGORICAL_COLS = ["contract", "internet_service", "payment_method",
                     "has_tech_support", "has_online_security"]
NUMERIC_COLS = ["tenure_months", "monthly_charges", "support_calls", "senior_citizen"]
RANDOM_STATE = 42


def clean_data(filename):
    df = pd.read_csv(filename)
    raw_count = len(df)

    df = df.drop_duplicates(subset="customer_id")
    df["monthly_charges"] = pd.to_numeric(df["monthly_charges"], errors="coerce")
    df = df.dropna(subset=["monthly_charges", "has_tech_support"])

    clean_count = len(df)
    print("Cleaning:", raw_count, "raw ->", clean_count, "clean")

    return df.reset_index(drop=True)


def get_churn_rate_by_group(df, group_column):
    groups = df[group_column].unique()
    rates = {}

    for group in groups:
        subset = df[df[group_column] == group]
        churned = subset[subset["churn"] == "Yes"]
        rate = round(len(churned) / len(subset) * 100, 1)
        rates[group] = rate

    return rates


def run_eda(df, filename):
    churned = df[df["churn"] == "Yes"]
    churn_rate = round(len(churned) / len(df) * 100, 1)

    churn_by_contract = get_churn_rate_by_group(df, "contract")
    churn_by_support_calls = get_churn_rate_by_group(df, "support_calls")

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    axes[0, 0].bar(list(churn_by_contract.keys()), list(churn_by_contract.values()), color="#C44E52")
    axes[0, 0].set_title("Churn Rate by Contract Type (%)")
    axes[0, 0].tick_params(axis="x", rotation=20)

    retained_tenure = df[df["churn"] == "No"]["tenure_months"]
    churned_tenure = df[df["churn"] == "Yes"]["tenure_months"]
    axes[0, 1].hist([retained_tenure, churned_tenure], label=["Retained", "Churned"],
                     bins=20, color=["#4C72B0", "#C44E52"])
    axes[0, 1].set_title("Tenure Distribution by Churn Status")
    axes[0, 1].set_xlabel("Tenure (months)")
    axes[0, 1].legend()

    retained_charges = df[df["churn"] == "No"]["monthly_charges"]
    churned_charges = df[df["churn"] == "Yes"]["monthly_charges"]
    axes[1, 0].hist([retained_charges, churned_charges], label=["Retained", "Churned"],
                     bins=20, color=["#4C72B0", "#C44E52"])
    axes[1, 0].set_title("Monthly Charges by Churn Status")
    axes[1, 0].set_xlabel("Monthly Charges ($)")
    axes[1, 0].legend()

    sorted_keys = sorted(churn_by_support_calls.keys())
    sorted_values = [churn_by_support_calls[k] for k in sorted_keys]
    axes[1, 1].bar(sorted_keys, sorted_values, color="#8172B2")
    axes[1, 1].set_title("Churn Rate by Support Calls (%)")
    axes[1, 1].set_xlabel("Number of Support Calls")

    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

    print("Overall churn rate:", churn_rate, "%")

    result = {"churn_rate": churn_rate, "churn_by_contract": churn_by_contract}
    return result


def prepare_features(df):
    df_encoded = df.copy()

    for col in CATEGORICAL_COLS:
        encoder = LabelEncoder()
        df_encoded[col] = encoder.fit_transform(df_encoded[col])

    X = df_encoded[NUMERIC_COLS + CATEGORICAL_COLS]
    y = (df_encoded["churn"] == "Yes").astype(int)

    return X, y


def evaluate_model(model, model_name, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    result = {
        "model": model,
        "accuracy": round(accuracy_score(y_test, predictions) * 100, 1),
        "precision": round(precision_score(y_test, predictions) * 100, 1),
        "recall": round(recall_score(y_test, predictions) * 100, 1),
        "f1": round(f1_score(y_test, predictions) * 100, 1),
        "roc_auc": round(roc_auc_score(y_test, probabilities), 3),
        "confusion_matrix": confusion_matrix(y_test, predictions).tolist()
    }

    print(model_name, "-> accuracy:", result["accuracy"], "recall:", result["recall"], "roc_auc:", result["roc_auc"])

    return result


def train_and_evaluate_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    results = {}

    model1 = LogisticRegression(max_iter=1000)
    results["Logistic Regression"] = evaluate_model(model1, "Logistic Regression", X_train, X_test, y_train, y_test)

    model2 = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE)
    results["Random Forest"] = evaluate_model(model2, "Random Forest", X_train, X_test, y_train, y_test)

    majority_share = y.value_counts(normalize=True).max()
    baseline_accuracy = round(majority_share * 100, 1)
    print("Baseline accuracy:", baseline_accuracy)

    return results, baseline_accuracy, list(X.columns)


def get_feature_importance(results, feature_names):
    rf_model = results["Random Forest"]["model"]
    importance_values = rf_model.feature_importances_

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importance_values
    })

    importance_df = importance_df.sort_values("importance", ascending=False)

    return importance_df


def plot_model_comparison(results, importance_df, filename):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    model_names = list(results.keys())
    metrics = ["accuracy", "precision", "recall", "f1"]
    x_positions = np.arange(len(metrics))
    width = 0.35

    for i in range(len(model_names)):
        name = model_names[i]
        values = [results[name][m] for m in metrics]
        axes[0].bar(x_positions + i * width, values, width, label=name)

    axes[0].set_xticks(x_positions + width / 2)
    axes[0].set_xticklabels(["Accuracy", "Precision", "Recall", "F1"])
    axes[0].set_title("Model Comparison")
    axes[0].set_ylabel("Score (%)")
    axes[0].legend()

    axes[1].barh(importance_df["feature"], importance_df["importance"], color="#55A868")
    axes[1].set_title("Feature Importance (Random Forest)")
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


def find_best_model(results):
    best_name = None
    best_score = -1

    for name in results:
        if results[name]["roc_auc"] > best_score:
            best_score = results[name]["roc_auc"]
            best_name = name

    return best_name


def write_report(eda, results, baseline_accuracy, importance_df, filename):
    best_model_name = find_best_model(results)
    best = results[best_model_name]
    top_features = importance_df.head(3)["feature"].tolist()

    file = open(filename, "w")

    file.write("CAPSTONE PROJECT 1: CUSTOMER CHURN PREDICTION\n")
    file.write("=" * 50 + "\n\n")

    file.write("1. DATA CLEANING AND EDA SUMMARY\n")
    file.write("-" * 40 + "\n")
    file.write("Overall churn rate: " + str(eda["churn_rate"]) + "%\n")
    file.write("Churn rate by contract type:\n")
    for contract in eda["churn_by_contract"]:
        file.write("  - " + contract + ": " + str(eda["churn_by_contract"][contract]) + "%\n")

    file.write("\n2. SUPERVISED ML MODEL RESULTS\n")
    file.write("-" * 40 + "\n")
    file.write("Baseline (always predict majority class): " + str(baseline_accuracy) + "%\n\n")

    for name in results:
        r = results[name]
        file.write(name + ": Accuracy=" + str(r["accuracy"]) + "%, Precision=" + str(r["precision"]))
        file.write("%, Recall=" + str(r["recall"]) + "%, F1=" + str(r["f1"]))
        file.write("%, ROC-AUC=" + str(r["roc_auc"]) + "\n")

    file.write("\nBest model: " + best_model_name + " (ROC-AUC = " + str(best["roc_auc"]) + ")\n")
    file.write("Confusion matrix (rows=actual, cols=predicted [No, Yes]): " + str(best["confusion_matrix"]) + "\n")

    file.write("\n3. CHURN RISK ANALYSIS\n")
    file.write("-" * 40 + "\n")
    file.write("Top churn risk factors: " + ", ".join(top_features) + "\n")

    file.write("\n4. BUSINESS RECOMMENDATIONS\n")
    file.write("-" * 40 + "\n")
    file.write("- Target month-to-month customers with incentives to switch to longer contracts.\n")
    file.write("- Flag customers with 3+ support calls for proactive outreach.\n")
    file.write("- Bundle free tech support and online security for at-risk new customers.\n")
    file.write("- Prioritize retention budget using the model's churn probability score.\n")
    file.write("- Recall is " + str(best["recall"]) + "%, so pair the model with other early-warning signals.\n")

    file.close()


def main():
    df = clean_data("customer_churn_raw.csv")
    df.to_csv("clean_customer_churn.csv", index=False)

    eda = run_eda(df, "churn_eda_charts.png")

    X, y = prepare_features(df)
    results, baseline_accuracy, feature_names = train_and_evaluate_models(X, y)

    importance_df = get_feature_importance(results, feature_names)
    plot_model_comparison(results, importance_df, "churn_model_comparison.png")

    write_report(eda, results, baseline_accuracy, importance_df, "churn_capstone_report.txt")

    print("Capstone Project 1 complete.")
    print("Files saved: clean_customer_churn.csv, churn_eda_charts.png, churn_model_comparison.png, churn_capstone_report.txt")


if __name__ == "__main__":
    main()

Cleaning: 1009 raw -> 940 clean
Overall churn rate: 38.1 %
Logistic Regression -> accuracy: 67.6 recall: 36.1 roc_auc: 0.676
Random Forest -> accuracy: 64.4 recall: 34.7 roc_auc: 0.674
Baseline accuracy: 61.9
Capstone Project 1 complete.
Files saved: clean_customer_churn.csv, churn_eda_charts.png, churn_model_comparison.png, churn_capstone_report.txt
